# Exercise 1.10 — Schur-Weyl Decomposition for $k = 2$

**Chapter 1: Mathematical Foundations** &nbsp;|&nbsp; *Section 1.8: Representation Theory (Schur-Weyl Duality)*

---

## Motivation

The **SWAP operator** $F$ is the fundamental building block of Schur-Weyl duality — the decomposition of $(\mathbb{C}^D)^{\otimes k}$ under the joint action of $U(D)$ and the symmetric group $S_k$.  For $k = 2$, the decomposition is elementary: the two-copy space splits into symmetric and antisymmetric subspaces.  The resulting **SWAP trick** $\mathrm{Tr}(AB) = \mathrm{Tr}[(A \otimes B) F]$ is the workhorse identity behind purity estimation, shadow tomography, and Weingarten calculus throughout this book.

## Exercise Statement

The SWAP operator $F$ on $\mathbb{C}^D \otimes \mathbb{C}^D$ exchanges tensor factors: $F|a,b\rangle = |b,a\rangle$.

**(a)** Prove $\mathrm{Tr}(F) = D$ and $\mathrm{Tr}(F^2) = D^2$.

**(b)** Show $\dim(\mathrm{Sym}^2) = D(D+1)/2$ and $\dim(\wedge^2) = D(D-1)/2$, and verify $\dim(\mathrm{Sym}^2) + \dim(\wedge^2) = D^2$.

**(c)** For $\rho = I/D$ (maximally mixed), verify the SWAP trick: $\mathrm{Tr}(\rho^2) = \mathrm{Tr}[(\rho \otimes \rho) F] = 1/D$.

## Solution

### Part (a): Traces of the SWAP operator

In the computational basis, the matrix elements of $F$ are:

$$
F_{(a,b),(c,d)} = \delta_{a,d}\,\delta_{b,c}.
$$

**Trace of $F$:**

$$
\mathrm{Tr}(F) = \sum_{a,b} F_{(a,b),(a,b)} = \sum_{a,b} \delta_{a,b}\,\delta_{b,a} = \sum_{a,b} \delta_{a,b} = D.
$$

Only diagonal elements $(a = b)$ contribute.

**Trace of $F^2$:** Since $F$ is an involution ($F^2 = I$, because swapping twice returns the original), $\mathrm{Tr}(F^2) = \mathrm{Tr}(I_{D^2}) = D^2$.

### Part (b): Symmetric and antisymmetric subspace dimensions

Since $F^2 = I$, the eigenvalues of $F$ are $\pm 1$.  The projectors onto the eigenspaces are:

$$
P_+ = \frac{I + F}{2} \quad (\text{symmetric}), \qquad P_- = \frac{I - F}{2} \quad (\text{antisymmetric}).
$$

Their dimensions (traces of the projectors):

$$
\dim(\mathrm{Sym}^2) = \mathrm{Tr}(P_+) = \frac{D^2 + D}{2} = \frac{D(D+1)}{2},
$$

$$
\dim(\wedge^2) = \mathrm{Tr}(P_-) = \frac{D^2 - D}{2} = \frac{D(D-1)}{2}.
$$

**Check:** $\frac{D(D+1)}{2} + \frac{D(D-1)}{2} = \frac{D(D+1+D-1)}{2} = D^2$. $\quad \checkmark$

For qubits ($D = 2$): $\mathrm{Sym}^2$ is the 3-dimensional triplet space, $\wedge^2$ is the 1-dimensional singlet.

### Part (c): The SWAP trick

The **SWAP trick** is the identity:

$$
\mathrm{Tr}(AB) = \mathrm{Tr}\bigl[(A \otimes B)\, F\bigr].
$$

**Proof:** In components,

$$
\mathrm{Tr}[(A \otimes B) F] = \sum_{a,b} (A \otimes B)_{(a,b),(c,d)}\, F_{(c,d),(a,b)} = \sum_{a,b} A_{a,b}\, B_{b,a} = \mathrm{Tr}(AB).
$$

**Application:** Setting $A = B = \rho = I/D$:

$$
\mathrm{Tr}(\rho^2) = \mathrm{Tr}[(\rho \otimes \rho) F] = \frac{1}{D^2}\mathrm{Tr}(F) = \frac{D}{D^2} = \frac{1}{D}. \quad \checkmark
$$

This confirms that the purity of the maximally mixed state equals $1/D$ — it becomes arbitrarily small as the Hilbert space dimension grows.

---
## Symbolic Verification (SymPy)

In [ ]:
import sympy as sp

for D in [2, 3, 4]:
    F = sp.zeros(D**2)
    for a in range(D):
        for b in range(D):
            row = a*D + b
            col = b*D + a
            F[row, col] = 1
    
    assert sp.trace(F) == D
    assert sp.trace(F*F) == D**2
    assert F*F == sp.eye(D**2)
    
    P_sym = (sp.eye(D**2) + F) / 2
    P_anti = (sp.eye(D**2) - F) / 2
    print(f"D={D}: Tr(F)={sp.trace(F)}, dim(Sym²)={sp.trace(P_sym)}, "
          f"dim(∧²)={sp.trace(P_anti)}, sum={sp.trace(P_sym)+sp.trace(P_anti)}")

---
## Numerical Verification

In [ ]:
import numpy as np

for D in [2, 3, 4, 5]:
    # Build SWAP
    F = np.zeros((D**2, D**2))
    for a in range(D):
        for b in range(D):
            F[a*D+b, b*D+a] = 1
    
    # SWAP trick: Tr(ρ²) via F
    rho = np.eye(D) / D
    purity_direct = np.trace(rho @ rho)
    purity_swap = np.trace(np.kron(rho, rho) @ F)
    
    assert np.isclose(purity_direct, 1/D)
    assert np.isclose(purity_swap, 1/D)
    
    # Also verify for a random pure state
    psi = np.random.randn(D) + 1j*np.random.randn(D)
    psi /= np.linalg.norm(psi)
    rho_pure = np.outer(psi, psi.conj())
    assert np.isclose(np.trace(np.kron(rho_pure, rho_pure) @ F), 1.0)
    
    print(f"D={D}: Tr(ρ²)_mixed = {purity_direct:.4f} = 1/{D}, "
          f"Tr(ρ²)_pure = {np.trace(rho_pure @ rho_pure).real:.4f} = 1")

print("\nSWAP trick verified for mixed and pure states. ✓")